In [1]:
import numpy as np
import os
import pandas as pd
import glob
import torch
import torchaudio
import soundfile as sf
import subprocess
import shutil
import scipy
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import norm
from speechbrain.inference.speaker import EncoderClassifier
from collections import defaultdict
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from speechbrain.lobes.features import Fbank
import torchaudio.transforms as T

In [12]:
file_path = '/add/your/path/here/'

In [16]:
wav_copied = False
exp1 = False
exp2 = True
wav2vec_final_layer = False

In [ ]:
# Set the base input and output directories
base_dir = '/add/your/path/here/'
output_dir = '/add/your/path/here/'

save_dir = '/add/your/path/here/'

if exp1: 
    wav_filepath_dir =  '/add/your/path/here/'
else: 
    wav_filepath_dir =  '/add/your/path/here/'


In [2]:
# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

dialect_regions = ['DR1', 'DR2', 'DR3', 'DR4', 'DR5', 'DR6', 'DR7', 'DR8']

# walk through dialect regions (e.g., DR1, DR2, ...)
#for dialect_region in os.listdir(base_dir):
if wav_copied:
    for dialect_region in dialect_regions:
        #print(dialect_region)
        region_path = os.path.join(base_dir, dialect_region)
        if os.path.isdir(region_path):
            # Walk through speaker folders (e.g., FDML0, MJEB1, ...)
            for speaker in os.listdir(region_path):
                speaker_path = os.path.join(region_path, speaker)
                if os.path.isdir(speaker_path):
                    sa2_path = os.path.join(speaker_path, "SA2.WAV")
                    if os.path.isfile(sa2_path):
                        new_filename = f"{speaker.lower()}_sa2.WAV"
                        dest_path = os.path.join(save_dir, new_filename)
                        shutil.copyfile(sa2_path, dest_path)
                        print(f"Copied {sa2_path} to {dest_path}")
else:
    print("wav files already in place")

In [70]:
if wav_copied: 
    sox_path = "/opt/homebrew/bin/sox" #replace with the output of which sox.
    
    # Convert each file using sox
    for filepath in wav_files:
        filename = os.path.basename(filepath)
        output_path = os.path.join(wav_filepath_dir, filename)
        #subprocess.run(["sox", filepath, output_path])
        subprocess.run([sox_path, filepath, output_path], check=True)
        #print(f"Converted: {filename}")
else: 
    print("conversion using sox already done")

conversion using sox already done


In [3]:
if wav_copied: 
    # Rename all .WAV to .wav
    for filepath in glob.glob(os.path.join(wav_filepath_dir, "*.WAV")):
        new_path = filepath.replace(".WAV", ".wav")
        os.rename(filepath, new_path)
        print(f"Renamed: {filepath} -> {new_path}")
else: 
    print("already renamed .WAV to .wav")

In [73]:
# Define helper function to load audio
def load_audio_with_soundfile(path):
    data, samplerate = sf.read(path)
    waveform = torch.tensor(data).float().unsqueeze(0)  # shape: [1, num_samples]
    return waveform, samplerate

In [32]:

# Manual fbank extractor with 80 Mel bins, no deltas, no context
fbank_transform = T.MelSpectrogram(
    sample_rate=16000,
    n_fft=400,
    win_length=400,
    hop_length=160,
    n_mels=80,
    power=1.0
)

amplitude_to_db = T.AmplitudeToDB()

def extract_fbank_features(waveform):
    with torch.no_grad():
        mel_spec = fbank_transform(waveform)  # [1, 80, time]
        log_mel = amplitude_to_db(mel_spec)   # convert to log scale (dB)
    return log_mel

wav_dirs = wav_filepath_dir
wavs_files = [f for f in os.listdir(wav_filepath_dir) if f.endswith('.wav')]
    

## wav2vec

In [4]:
# Paths
wav_dirs = wav_filepath_dir
wavs_files = [f for f in os.listdir(wav_dirs) if f.endswith('.wav')]
# Load Wav2Vec2
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")
model.eval()

In [78]:
def load_audio_with_soundfile_for_wav2vec(path):
    data, samplerate = sf.read(path)
    if len(data.shape) > 1:  # stereo -> mono
        data = data.mean(axis=1)
    waveform = torch.tensor(data).float()  # shape: [num_samples]
    return waveform, samplerate

In [79]:
# Storage
embeddings_list = []

for fname in wavs_files:
    path = os.path.join(wav_dirs, fname)
    waveform, samplerate = load_audio_with_soundfile_for_wav2vec(path)

    # Resample to 16kHz if necessary
    if samplerate != 16000:
        raise ValueError(f"{fname} is not 16kHz. Please resample before feeding to wav2vec2.")

    # Process input
    inputs = processor(waveform, sampling_rate=16000, return_tensors="pt")
    
    # Get hidden states (shape: [1, time_steps, hidden_size])
    with torch.no_grad():
        outputs = model(**inputs)
        hidden_states = outputs.last_hidden_state.squeeze(0)  # remove batch dim

    # Aggregate over time (e.g., mean pooling)
    embedding = hidden_states.mean(dim=0)  # shape: [hidden_size]
    
    embeddings_list.append({
        "filename": fname,
        **{f"dim_{i}": val.item() for i, val in enumerate(embedding)}
    })

# Create DataFrame
wav2vec_embeddings = pd.DataFrame(embeddings_list)

In [81]:
# Compute L2 norm of wav2vec embeddings (excluding filename column)
wav2vec_l2_df = pd.DataFrame({
    'filename': wav2vec_embeddings['filename'],
    'l2_norm_wav2vec': wav2vec_embeddings.iloc[:, 1:].apply(lambda row: np.linalg.norm(row), axis=1)
})

In [82]:
if wav2vec_final_layer:
    print('already loaded the final layer of wav2vec')
else:
    # Storage
    embeddings_list = []
    
    for fname in wavs_files:
        path = os.path.join(wav_dirs, fname)
        waveform, samplerate = load_audio_with_soundfile_for_wav2vec(path)
    
        if samplerate != 16000:
            raise ValueError(f"{fname} is not 16kHz. Please resample before feeding to wav2vec2.")
    
        # Tokenize
        inputs = processor(waveform, sampling_rate=16000, return_tensors="pt")
        
        # Extract all hidden states
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            all_hidden_states = outputs.hidden_states  # Tuple of [layer_0, ..., layer_N]
    
        # L2 norm for each layer (after mean pooling)
        layer_norms = {}
        for i, hs in enumerate(all_hidden_states):
            hs_pooled = hs.squeeze(0).mean(dim=0)  # shape: [hidden_dim]
            l2_norm = torch.norm(hs_pooled, p=2).item()
            layer_norms[f'L2_norm_layer_{i}'] = l2_norm
    
        embeddings_list.append({
            'filename': fname,
            **layer_norms
        })
    
    # Create DataFrame
    wav2vec_l2norms = pd.DataFrame(embeddings_list)

## Compute memorability of the Revsine data 

In [84]:
revsine_data_path = '/add/your/path/here/'

In [85]:
def load_csv_file(filename):
    df = pd.read_csv(filename)
    return df

In [5]:
if exp1:
    data = revsine_data_path + '/experiment_1_data.csv'
    print("loading data for experiment 1")
else exp2:
    data = revsine_data_path + '/experiment_2_data.csv'
    print("loading data for experiment 2")

In [87]:
revsine_data = load_csv_file(data)

In [90]:
# Normalize file names by removing extensions; so we can avoid duplicates 
revsine_data['AudioClipSequence'] = revsine_data['AudioClipSequence'].str.replace(r'\.(wav|mp3)', '', regex=True)

In [91]:
# Step 1: Expand the sequences into one row per stimulus
expanded_rows = []
for _, row in revsine_data.iterrows():
    subject_id = row['SubjectID']
    stimuli = [s.rsplit('.', 1)[0] for s in row['AudioClipSequence'].split(';')]
    types = list(map(int, row['StimulusTypeSequence'].split(';')))
    performances = list(map(int, row['PerformanceSequence'].split(';')))

    for stim, stim_type, perf in zip(stimuli, types, performances):
        expanded_rows.append({
            'SubjectID': subject_id,
            'Stimulus': stim,
            'StimulusType': stim_type,
            'Performance': perf
        })

In [92]:
df_expanded = pd.DataFrame(expanded_rows)

In [93]:
# Step 2: Calculate Hit Rate (Target Repeat = 2 and Performance = 11)
target_repeat = df_expanded[df_expanded['StimulusType'] == 2]
hits = target_repeat[target_repeat['Performance'] == 11].groupby('Stimulus').size()
total_repeats = target_repeat.groupby('Stimulus').size()
hit_rate = (hits / total_repeats).fillna(0)

In [94]:
# Step 3: Calculate False Alarm Rate (Initial Target = 1 and Performance = 13)
initial_targets = df_expanded[df_expanded['StimulusType'] == 1]
false_alarms = initial_targets[initial_targets['Performance'] == 13].groupby('Stimulus').size()
total_initials = initial_targets.groupby('Stimulus').size()
fa_rate = (false_alarms / total_initials).fillna(0)

In [95]:
# Combine into a single DataFrame
final_df = pd.DataFrame({
    'Stimulus': sorted(set(hit_rate.index).union(fa_rate.index)),
})

In [96]:
# Add hit_rate and fa_rate, reindexing to ensure alignment
final_df['hit_rate'] = final_df['Stimulus'].map(hit_rate).fillna(0)
final_df['fa_rate'] = final_df['Stimulus'].map(fa_rate).fillna(0)

# Clip rates to avoid infinite d'
epsilon = 1e-5
final_df['hit_rate'] = final_df['hit_rate'].clip(epsilon, 1 - epsilon)
final_df['fa_rate'] = final_df['fa_rate'].clip(epsilon, 1 - epsilon)

# Compute d'
final_df['d_prime'] = norm.ppf(final_df['hit_rate']) - norm.ppf(final_df['fa_rate'])

## Compute Memorability and L2 norm correlation

In [105]:
wav2vec_l2_df['base_id'] = wav2vec_l2_df['filename'].str[:5]
final_df['base_id'] = final_df['Stimulus'].str[:5]
final_df = final_df.merge(wav2vec_l2_df[['base_id', 'l2_norm_wav2vec']], on='base_id', how='left')
final_df.drop(columns='base_id', inplace=True)
revsine_memo_df = final_df.dropna(subset=['l2_norm_wav2vec'])

In [104]:
if wav2vec_final_layer: 
    wav2vec_l2_df['base_id'] = wav2vec_l2_df['filename'].str[:5]
    final_df['base_id'] = final_df['Stimulus'].str[:5]
    final_df = final_df.merge(wav2vec_l2_df[['base_id', 'l2_norm_wav2vec']], on='base_id', how='left')
    final_df.drop(columns='base_id', inplace=True)
    revsine_memo_df = final_df.dropna(subset=['l2_norm_wav2vec'])
else: 

    wav2vec_l2norms['base_id'] = wav2vec_l2norms['filename'].str[:5]    
    final_df['base_id'] = final_df['Stimulus'].str[:5]
    final_df = final_df.merge(wav2vec_l2norms[['base_id', 'L2_norm_layer_0', 'L2_norm_layer_1', 'L2_norm_layer_2', 'L2_norm_layer_3', 'L2_norm_layer_4', 'L2_norm_layer_5', 'L2_norm_layer_6', 'L2_norm_layer_7', 'L2_norm_layer_8', 'L2_norm_layer_9', 'L2_norm_layer_10', 'L2_norm_layer_11', 'L2_norm_layer_12']], on='base_id', how='left')

    final_df.drop(columns='base_id', inplace=True)
    revsine_memo_df = final_df.dropna(subset=['L2_norm_layer_0'])


In [6]:
correlation_coefficient, p_value = scipy.stats.spearmanr(revsine_memo_df['L2_norm_layer_11'], revsine_memo_df['d_prime'])
print("Pearson correlation coefficient:", correlation_coefficient)
print("P-value recall:", p_value)


In [108]:
revsine_memo_df.to_csv('revsine_exp2_memorability_wav2vecL2norm_dav.csv', index=False)  